# Dataset and DataLoader

Dataset and DataLoader are core abstracitons in Pytorch that decouple how you define your data from how you efficiently iterate over it in training loops.

In simple terms,

Dataset -> does work that is realted to loading the data

DataLoader -> converting that data into batches

#$Dataset$ $Class$

The dataset class is essentially a blueprint.

When you create a custom Dataset, you decide how data is loaded and returned.

it defines:

1. __init__() : tells how data should be loaded
2. __len__() : returns the total number of samples
3. __getitem()__(index) : returns the data (and label) at the given index.

#$DataLoader$ $Class$

The DataLoader wraps a Dataset and handles batching, shuffling, and parallel loading for you.


DataLoader Control Flow:

1. At the start of each epoch, the DataLoader (if shuffle = True) shuffles incides (using a sampler).

2. It divides the indices into chinks of batch_size.

3. for each index in the chunk, data samples are fetched from the Dataset object

4. The samples are then collected and combined into a batch (using collate_fn).

5. The batch is returned ot the main training loop.

In [1]:
from sklearn.datasets import make_classification
import torch

In [2]:
# creating a synthetic dataset
X, y = make_classification(
    n_samples = 10,
    n_features = 2,
    n_informative = 2,
    n_redundant = 0,
    n_classes = 2,
    random_state = 42
)

In [3]:
X

array([[ 1.06833894, -0.97007347],
       [-1.14021544, -0.83879234],
       [-2.8953973 ,  1.97686236],
       [-0.72063436, -0.96059253],
       [-1.96287438, -0.99225135],
       [-0.9382051 , -0.54304815],
       [ 1.72725924, -1.18582677],
       [ 1.77736657,  1.51157598],
       [ 1.89969252,  0.83444483],
       [-0.58723065, -1.97171753]])

In [4]:
X.shape

(10, 2)

In [5]:
y

array([1, 0, 0, 0, 0, 1, 1, 1, 1, 0])

In [6]:
y.shape

(10,)

In [7]:
# converting the data from numpy arrays to tensors
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)

In [8]:
X

tensor([[ 1.0683, -0.9701],
        [-1.1402, -0.8388],
        [-2.8954,  1.9769],
        [-0.7206, -0.9606],
        [-1.9629, -0.9923],
        [-0.9382, -0.5430],
        [ 1.7273, -1.1858],
        [ 1.7774,  1.5116],
        [ 1.8997,  0.8344],
        [-0.5872, -1.9717]])

In [9]:
y

tensor([1., 0., 0., 0., 0., 1., 1., 1., 1., 0.])

In [10]:
from torch.utils.data import Dataset, DataLoader

In [11]:
class CustomDataset(Dataset):

  def __init__(self, features, labels):
    self.features = features
    self.labels = labels

  def __len__(self):
    return self.features.shape[0]

  def __getitem__(self,index):
    return self.features[index], self.labels[index]

In [12]:
dataset = CustomDataset(X,y)

In [13]:
len(dataset)

10

In [14]:
print(dataset[3])
print(X)

(tensor([-0.7206, -0.9606]), tensor(0.))
tensor([[ 1.0683, -0.9701],
        [-1.1402, -0.8388],
        [-2.8954,  1.9769],
        [-0.7206, -0.9606],
        [-1.9629, -0.9923],
        [-0.9382, -0.5430],
        [ 1.7273, -1.1858],
        [ 1.7774,  1.5116],
        [ 1.8997,  0.8344],
        [-0.5872, -1.9717]])


In [15]:
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

In [16]:
for batch_features, batch_labels in dataloader:
  print(batch_features)
  print(batch_labels)
  print("-"*50)

tensor([[ 1.0683, -0.9701],
        [-1.1402, -0.8388]])
tensor([1., 0.])
--------------------------------------------------
tensor([[-0.9382, -0.5430],
        [ 1.8997,  0.8344]])
tensor([1., 1.])
--------------------------------------------------
tensor([[-2.8954,  1.9769],
        [ 1.7273, -1.1858]])
tensor([0., 1.])
--------------------------------------------------
tensor([[ 1.7774,  1.5116],
        [-0.5872, -1.9717]])
tensor([1., 0.])
--------------------------------------------------
tensor([[-1.9629, -0.9923],
        [-0.7206, -0.9606]])
tensor([0., 0.])
--------------------------------------------------


#$Sampler$

In pytorch, the sampler in the DataLoader determins the strategy for selecting samples from the dataset during data loading. It controls how indices of the dataset are drawn for each batch.

## Types of Samplers

Pytorch provides several predefined samplers, and you can also create a custom sampler based on the use case.

1. SequentialSampler:
  - Samples elements sequentially, in the order they appear in the dataset
  - Default when shuffle = False

2. RandomSampler:
  - Samples elements randomly without replacement
  - Default when shuffle = True

3. CustomSampler:
  - This is very domain specific.
  
  It can be used in a scenario for example: out dataset is imbalanced and the with class 1 as 95% and the class 2 as 5%. then while creating the batches, you also want to make sure that this same distrubition stays in the batches.